# Notebook 29 — Failure Modes

**Continual Learning in Context: RML Extension for CL-BENCH**

Notebook 23 analyzed revision after drift.

Notebook 29 asks:

> When does revision fail?

This notebook closes the first RML notebook sequence by classifying continual-learning failure modes.

Failure modes:

- **strong_revision** — drift is revised successfully
- **weak_revision** — some recovery, but incomplete
- **no_revision** — no recovery after drift
- **slow_revision** — recovery occurs, but lag is high
- **oscillating_revision** — repeated gain up/down cycles
- **regressive_revision** — recovery damages prior context

Outputs:

- `results/29_revision_failures.csv`
- `results/29_failure_mode_by_variant.csv`
- `results/29_forgetting_cost_matrix.csv`
- `results/29_failure_mode_summary.json`
- `figures/29_failure_modes_by_variant.png`
- `figures/29_adaptation_lag_distribution.png`
- `figures/29_oscillation_timeline.png`
- `figures/29_stability_revision_plane.png`
- `figures/29_forgetting_cost_matrix.png`
- `results/29_failure_modes_artifacts.zip`

## 0. Bootstrap Repo Path

This cell works locally or in Colab.

If opened directly in Colab and the repo is missing, it clones:

```text
https://github.com/thinkthoughts/continual-learning-bench-rml
```

In [ ]:
from pathlib import Path
import sys
import os
import subprocess

REPO_URL = "https://github.com/thinkthoughts/continual-learning-bench-rml.git"
REPO_NAME = "continual-learning-bench-rml"

def running_in_colab() -> bool:
    try:
        import google.colab  # type: ignore
        return True
    except Exception:
        return False

def find_rml_root(start: Path) -> Path | None:
    start = start.resolve()
    for base in [start, *start.parents]:
        if (base / "src" / "gain.py").exists():
            return base
        if (base / "rml" / "src" / "gain.py").exists():
            return base / "rml"
        if (base / REPO_NAME / "rml" / "src" / "gain.py").exists():
            return base / REPO_NAME / "rml"
    return None

cwd = Path.cwd().resolve()
RML_ROOT = find_rml_root(cwd)

if RML_ROOT is None and running_in_colab():
    repo_path = Path("/content") / REPO_NAME

    if repo_path.exists():
        print("Repo already exists. Pulling latest changes...")
        subprocess.run(["git", "-C", str(repo_path), "pull"], check=False)
    else:
        print("Cloning repo...")
        subprocess.run(["git", "clone", REPO_URL, str(repo_path)], check=True)

    RML_ROOT = repo_path / "rml"
    os.chdir(RML_ROOT)

elif RML_ROOT is not None:
    os.chdir(RML_ROOT)

else:
    raise RuntimeError(
        "Could not locate rml/src/gain.py. "
        "Run this notebook inside the repo, or in Colab allow the bootstrap cell to clone the repo."
    )

if str(RML_ROOT) not in sys.path:
    sys.path.insert(0, str(RML_ROOT))

RESULTS_DIR = RML_ROOT / "results"
FIGURES_DIR = RML_ROOT / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Running in Colab:", running_in_colab())
print("Current working directory:", Path.cwd())
print("RML root:", RML_ROOT)
print("results:", RESULTS_DIR)
print("figures:", FIGURES_DIR)

## 1. Imports

In [ ]:
import json
import zipfile
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.gain import compute_gain

print("Imports complete.")

## 2. Load Prior Artifacts

Notebook 29 prefers Notebook 23 outputs:

```text
results/23_revision_scores.csv
results/23_drift_recovery.csv
results/23_drift_candidates.csv
```

It also uses a trajectory from prior notebooks when available.

Fallback behavior:

If revision outputs are missing, Notebook 29 constructs a small fallback set of revision events so the failure-mode taxonomy can still run.

In [ ]:
revision_scores_path = RESULTS_DIR / "23_revision_scores.csv"
drift_recovery_path = RESULTS_DIR / "23_drift_recovery.csv"
drift_candidates_path = RESULTS_DIR / "23_drift_candidates.csv"

revision_source = None

if revision_scores_path.exists():
    revision_scores = pd.read_csv(revision_scores_path)
    revision_source = revision_scores_path
    print("Loaded:", revision_scores_path)
else:
    print("No Notebook 23 revision scores found. Creating fallback revision outcomes.")
    revision_scores = pd.DataFrame({
        "drift_instance": [5, 7, 9, 11],
        "variant": ["B", "B", "C", "C"],
        "gain_at_drift": [0.21, 0.25, 0.17, 0.31],
        "pre_drift_gain": [0.14, 0.28, 0.30, 0.23],
        "peak_future_gain": [0.30, 0.25, 0.35, 0.31],
        "peak_future_instance": [8, 7, 12, 11],
        "final_gain_in_variant": [0.30, 0.25, 0.35, 0.31],
        "recovery_gain": [0.09, 0.00, 0.18, 0.00],
        "final_recovery": [0.09, 0.00, 0.18, 0.00],
        "exceeded_pre_drift_gain": [True, False, True, True],
        "adaptation_lag_instances": [3, np.nan, 2, 0],
        "revision_label": [
            "partial_recovery",
            "no_recovery",
            "successful_revision",
            "no_recovery",
        ],
        "stale_risk_score": [0.5, 1.0, 2.5, 0.5],
        "lag_penalty": [0.06, 0.10, 0.04, 0.0],
        "revision_success_score": [0.03, -0.10, 0.14, 0.00],
        "revision_success_class": [
            "weak_revision",
            "failed_revision",
            "strong_revision",
            "failed_revision",
        ],
    })
    revision_source = "fallback_revision_outcomes"

# Load trajectory for oscillation and forgetting analyses.
trajectory_paths = [
    RESULTS_DIR / "19_stale_context_events.csv",
    RESULTS_DIR / "17_plasticity_states.csv",
    RESULTS_DIR / "13_context_stability.csv",
    RESULTS_DIR / "11_state_vs_stateless_instances.csv",
    RESULTS_DIR / "01_gain_signal_trajectory.csv",
    RESULTS_DIR / "00_context_gain.csv",
]

trajectory_source = None
df = None

for path in trajectory_paths:
    if path.exists():
        df = pd.read_csv(path)
        trajectory_source = path
        print("Loaded trajectory:", path)
        break

if df is None:
    print("No prior trajectory found. Creating fallback toy trajectory.")
    df = pd.DataFrame({
        "instance": np.arange(1, 13),
        "variant": [
            "A", "A", "A", "A",
            "B", "B", "B", "B",
            "C", "C", "C", "C",
        ],
        "stateful_reward": [
            0.18, 0.22, 0.28, 0.35,
            0.43, 0.48, 0.46, 0.52,
            0.40, 0.45, 0.55, 0.60,
        ],
        "stateless_reward": [
            0.18, 0.19, 0.20, 0.21,
            0.22, 0.20, 0.21, 0.22,
            0.23, 0.22, 0.24, 0.25,
        ],
    })
    trajectory_source = "fallback_trajectory"

if "gain" not in df.columns:
    if "state_advantage" in df.columns:
        df["gain"] = df["state_advantage"]
    else:
        df["gain"] = compute_gain(
            df["stateful_reward"].tolist(),
            df["stateless_reward"].tolist(),
        )

if "gain_delta" not in df.columns:
    df["gain_delta"] = df["gain"].diff().fillna(0.0)

if "cumulative_gain" not in df.columns:
    df["cumulative_gain"] = df["gain"].cumsum()

df = df.sort_values("instance").reset_index(drop=True)

revision_scores

## 3. Failure-Mode Definitions

Notebook 29 classifies revision outcomes using interpretable rules.

Rules:

```text
strong_revision:
    revision_success_score >= 0.10

slow_revision:
    recovery exists, but lag > 3

oscillating_revision:
    repeated sign changes in gain_delta near the event

regressive_revision:
    recovery coincides with degradation in another prior context

no_revision:
    recovery_gain <= 0

weak_revision:
    recovery exists, but score is below strong threshold
```

This taxonomy is intentionally simple. It is meant to make failure modes visible before a more formal metric is introduced.

## 4. Detect Oscillation

In [ ]:
oscillation_rows = []

for _, event in revision_scores.iterrows():
    inst = int(event["drift_instance"])
    variant = event["variant"]

    window = df[
        (df["instance"] >= inst - 2)
        & (df["instance"] <= inst + 3)
    ].copy()

    deltas = window["gain_delta"].tolist()
    signs = [
        1 if x > 0 else -1 if x < 0 else 0
        for x in deltas
    ]

    sign_changes = 0
    prev = None
    for s in signs:
        if s == 0:
            continue
        if prev is not None and s != prev:
            sign_changes += 1
        prev = s

    oscillation_rows.append({
        "drift_instance": inst,
        "variant": variant,
        "window_size": int(len(window)),
        "gain_delta_sign_changes": int(sign_changes),
        "oscillation_flag": bool(sign_changes >= 2),
    })

oscillation_df = pd.DataFrame(oscillation_rows)
oscillation_df

## 5. Forgetting Cost Matrix

Forgetting cost is approximated as the loss in mean gain for a prior variant after a later drift event.

This is a lightweight proxy:

\[
\Delta g =
\text{mean gain after drift}
-
\text{mean gain before drift}
\]

Negative values suggest regression.

In [ ]:
variants = df["variant"].drop_duplicates().tolist()
forgetting_matrix = pd.DataFrame(
    0.0,
    index=variants,
    columns=variants,
)

forgetting_rows = []

for _, event in revision_scores.iterrows():
    drift_variant = event["variant"]
    drift_instance = int(event["drift_instance"])

    for prior_variant in variants:
        before = df[
            (df["variant"] == prior_variant)
            & (df["instance"] < drift_instance)
        ]["gain"]

        after = df[
            (df["variant"] == prior_variant)
            & (df["instance"] >= drift_instance)
        ]["gain"]

        if len(before) == 0 or len(after) == 0:
            delta = 0.0
        else:
            delta = float(after.mean() - before.mean())

        forgetting_matrix.loc[prior_variant, drift_variant] = delta

        forgetting_rows.append({
            "prior_variant": prior_variant,
            "drift_variant": drift_variant,
            "drift_instance": drift_instance,
            "forgetting_cost": delta,
        })

forgetting_costs = pd.DataFrame(forgetting_rows)
forgetting_matrix

## 6. Classify Revision Failure Modes

In [ ]:
failures = revision_scores.merge(
    oscillation_df,
    on=["drift_instance", "variant"],
    how="left",
)

# Add worst forgetting cost caused by this drift variant.
worst_forgetting = (
    forgetting_costs.groupby(["drift_variant", "drift_instance"])
    .agg(worst_forgetting_cost=("forgetting_cost", "min"))
    .reset_index()
    .rename(columns={"drift_variant": "variant"})
)

failures = failures.merge(
    worst_forgetting,
    on=["variant", "drift_instance"],
    how="left",
)

def classify_failure(row):
    score = float(row.get("revision_success_score", 0.0))
    recovery = float(row.get("recovery_gain", 0.0))
    lag = row.get("adaptation_lag_instances", np.nan)
    oscillating = bool(row.get("oscillation_flag", False))
    forgetting = float(row.get("worst_forgetting_cost", 0.0))

    if oscillating:
        return "oscillating_revision"

    if forgetting < -0.05:
        return "regressive_revision"

    if recovery <= 0:
        return "no_revision"

    if not pd.isna(lag) and lag > 3:
        return "slow_revision"

    if score >= 0.10:
        return "strong_revision"

    if score > 0:
        return "weak_revision"

    return "failed_revision"

failures["failure_mode"] = failures.apply(classify_failure, axis=1)

failures[[
    "variant",
    "drift_instance",
    "recovery_gain",
    "adaptation_lag_instances",
    "revision_success_score",
    "gain_delta_sign_changes",
    "worst_forgetting_cost",
    "failure_mode",
]]

## 7. Failure Modes by Variant

In [ ]:
failure_mode_by_variant = (
    failures.groupby(["variant", "failure_mode"])
    .size()
    .reset_index(name="count")
)

failure_mode_by_variant

## 8. Figure — Failure Modes by Variant

In [ ]:
pivot_failures = (
    failure_mode_by_variant
    .pivot(index="variant", columns="failure_mode", values="count")
    .fillna(0)
)

fig, ax = plt.subplots(figsize=(9, 5))

bottom = np.zeros(len(pivot_failures))

for column in pivot_failures.columns:
    values = pivot_failures[column].values
    ax.bar(pivot_failures.index, values, bottom=bottom, label=column)
    bottom += values

ax.set_title("Notebook 29: Revision Failure Modes by Variant")
ax.set_xlabel("Variant")
ax.set_ylabel("Count")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)

failure_modes_by_variant_path = FIGURES_DIR / "29_failure_modes_by_variant.png"
fig.tight_layout()
fig.savefig(failure_modes_by_variant_path, dpi=160)

failure_modes_by_variant_path

## 9. Figure — Adaptation Lag Distribution

In [ ]:
lag_values = failures["adaptation_lag_instances"].dropna()

fig, ax = plt.subplots(figsize=(8, 5))

if len(lag_values) > 0:
    ax.hist(lag_values, bins=range(0, int(lag_values.max()) + 3))
else:
    ax.text(0.5, 0.5, "No finite adaptation lags", ha="center", va="center")

ax.set_title("Notebook 29: Adaptation Lag Distribution")
ax.set_xlabel("Adaptation Lag")
ax.set_ylabel("Count")
ax.grid(True, axis="y", alpha=0.3)

lag_distribution_path = FIGURES_DIR / "29_adaptation_lag_distribution.png"
fig.tight_layout()
fig.savefig(lag_distribution_path, dpi=160)

lag_distribution_path

## 10. Figure — Oscillation Timeline

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    df["instance"],
    df["gain"],
    marker="o",
    label="gain",
)

for _, row in failures.iterrows():
    inst = int(row["drift_instance"])
    ax.axvline(inst, linestyle="--", linewidth=1)

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 29: Oscillation Timeline")
ax.set_xlabel("Instance")
ax.set_ylabel("Gain")
ax.legend()
ax.grid(True, alpha=0.3)

oscillation_timeline_path = FIGURES_DIR / "29_oscillation_timeline.png"
fig.tight_layout()
fig.savefig(oscillation_timeline_path, dpi=160)

oscillation_timeline_path

## 11. Figure — Stability–Revision Plane

This figure compares stale-risk pressure to revision outcome.

It is a final summary of the 19 → 23 → 29 sequence.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(
    failures["stale_risk_score"],
    failures["revision_success_score"],
    s=140,
)

for _, row in failures.iterrows():
    ax.annotate(
        f"{row['variant']}:{int(row['drift_instance'])}\n{row['failure_mode']}",
        (row["stale_risk_score"], row["revision_success_score"]),
        textcoords="offset points",
        xytext=(8, 8),
    )

ax.axhline(0, linewidth=1)
ax.set_title("Notebook 29: Stability–Revision Plane")
ax.set_xlabel("Stale Risk Score")
ax.set_ylabel("Revision Success Score")
ax.grid(True, alpha=0.3)

stability_revision_plane_path = FIGURES_DIR / "29_stability_revision_plane.png"
fig.tight_layout()
fig.savefig(stability_revision_plane_path, dpi=160)

stability_revision_plane_path

## 12. Figure — Forgetting Cost Matrix

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

matrix_values = forgetting_matrix.values
im = ax.imshow(matrix_values, aspect="auto")

ax.set_xticks(np.arange(len(forgetting_matrix.columns)))
ax.set_yticks(np.arange(len(forgetting_matrix.index)))

ax.set_xticklabels(forgetting_matrix.columns)
ax.set_yticklabels(forgetting_matrix.index)

for i in range(matrix_values.shape[0]):
    for j in range(matrix_values.shape[1]):
        ax.text(j, i, f"{matrix_values[i, j]:.2f}", ha="center", va="center")

ax.set_title("Notebook 29: Forgetting Cost Matrix")
ax.set_xlabel("Drift Variant")
ax.set_ylabel("Prior Variant")

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

forgetting_cost_matrix_path = FIGURES_DIR / "29_forgetting_cost_matrix.png"
fig.tight_layout()
fig.savefig(forgetting_cost_matrix_path, dpi=160)

forgetting_cost_matrix_path

## 13. Summary

In [ ]:
failure_counts = {
    mode: int(count)
    for mode, count in failures["failure_mode"].value_counts().items()
}

if len(failures) > 0:
    best = failures.loc[failures["revision_success_score"].idxmax()].to_dict()
    worst = failures.loc[failures["revision_success_score"].idxmin()].to_dict()
else:
    best = {}
    worst = {}

summary = {
    "revision_source": str(revision_source),
    "trajectory_source": str(trajectory_source),
    "total_revision_events": int(len(failures)),
    "failure_mode_counts": failure_counts,
    "mean_revision_success_score": float(failures["revision_success_score"].mean()),
    "mean_recovery_gain": float(failures["recovery_gain"].mean()),
    "mean_adaptation_lag": float(failures["adaptation_lag_instances"].dropna().mean())
        if len(failures["adaptation_lag_instances"].dropna()) > 0 else None,
    "mean_forgetting_cost": float(forgetting_costs["forgetting_cost"].mean()),
    "worst_forgetting_cost": float(forgetting_costs["forgetting_cost"].min()),
    "best_revision_variant": str(best.get("variant", "")),
    "best_revision_instance": int(best.get("drift_instance", -1)),
    "best_revision_success_score": float(best.get("revision_success_score", 0.0)),
    "worst_revision_variant": str(worst.get("variant", "")),
    "worst_revision_instance": int(worst.get("drift_instance", -1)),
    "worst_revision_success_score": float(worst.get("revision_success_score", 0.0)),
}

summary

## 14. Export Artifacts

In [ ]:
revision_failures_csv = RESULTS_DIR / "29_revision_failures.csv"
failure_mode_by_variant_csv = RESULTS_DIR / "29_failure_mode_by_variant.csv"
forgetting_cost_matrix_csv = RESULTS_DIR / "29_forgetting_cost_matrix.csv"
forgetting_costs_csv = RESULTS_DIR / "29_forgetting_costs.csv"
summary_path = RESULTS_DIR / "29_failure_mode_summary.json"
zip_path = RESULTS_DIR / "29_failure_modes_artifacts.zip"

failures.to_csv(revision_failures_csv, index=False)
failure_mode_by_variant.to_csv(failure_mode_by_variant_csv, index=False)
forgetting_matrix.to_csv(forgetting_cost_matrix_csv)
forgetting_costs.to_csv(forgetting_costs_csv, index=False)

summary_with_metadata = {
    **summary,
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "29_failure_modes.ipynb",
    "extension": "continual-learning-bench-rml",
    "repo": REPO_URL,
}

with open(summary_path, "w") as f:
    json.dump(summary_with_metadata, f, indent=2)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for path in [
        revision_failures_csv,
        failure_mode_by_variant_csv,
        forgetting_cost_matrix_csv,
        forgetting_costs_csv,
        summary_path,
        failure_modes_by_variant_path,
        lag_distribution_path,
        oscillation_timeline_path,
        stability_revision_plane_path,
        forgetting_cost_matrix_path,
    ]:
        z.write(path, arcname=path.name)

print("Saved artifacts:")
for path in [
    revision_failures_csv,
    failure_mode_by_variant_csv,
    forgetting_cost_matrix_csv,
    forgetting_costs_csv,
    summary_path,
    failure_modes_by_variant_path,
    lag_distribution_path,
    oscillation_timeline_path,
    stability_revision_plane_path,
    forgetting_cost_matrix_path,
    zip_path,
]:
    print("-", path)

## 15. Download Artifacts

In Colab, this downloads the zip.

Locally, the archive is saved under:

```text
rml/results/29_failure_modes_artifacts.zip
```

In [ ]:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Download helper is only active in Colab.")
    print("Artifact archive:", zip_path)

## 16. Notebook 29 Claim

Notebook 23 asked whether stale context can be revised.

Notebook 29 asks why revisions fail.

\[
\text{drift}
\rightarrow
\text{revision}
\rightarrow
\text{failure mode taxonomy}
\]

In RML terms:

> continual learning fails when a system cannot decide whether to retain, revise, or discard context.

This completes the first lab-report-init sequence:

\[
0 \rightarrow \{1,7,11,13,17,19,23,29\}
\]

The next step is to replace the toy trajectories with full CL-BENCH rollout logs.